## Lesson 7: Parameter-Efficient Fine-Tuning (LoRA)
### What you'll learn
- Why full fine-tuning is expensive and what LoRA does instead.
- Wrap a pLM with a LoRA adapter using the `peft` library (one config object).
- Print and understand the trainable-parameter fraction (<1% of total).
- Fine-tune only the adapters and compare accuracy/F1 to Lesson 3's full fine-tune.

### What LoRA does mathematically

A linear layer applies a frozen weight matrix $W \in \mathbb{R}^{d_\text{out} \times d_\text{in}}$.
Full fine-tuning learns an update $\Delta W$ of the *same* shape, so $W \leftarrow W + \Delta W$.
LoRA instead constrains $\Delta W$ to be **low-rank**, factoring it into two small matrices:

$$
\Delta W = B A,
\qquad
A \in \mathbb{R}^{r \times d_\text{in}},\;
B \in \mathbb{R}^{d_\text{out} \times r},\;
r \ll \min(d_\text{in}, d_\text{out}).
$$

The adapted forward pass, with the usual scaling factor $\alpha / r$, is

$$
h = W x + \frac{\alpha}{r}\, B A x ,
$$

where only $A$ and $B$ are trained and $W$ stays frozen. $A$ is initialised from a random
Gaussian and $B$ from zeros, so $\Delta W = 0$ at the start — the adapted model begins
identical to the base model.

**Parameter count.** A full update has $d_\text{out}\, d_\text{in}$ trainable entries; LoRA
has only $r\,(d_\text{in} + d_\text{out})$. With $r = 8$, $\alpha = 16$ and adapters on the
query/value projections of ESM-2 8M, the run below trains **164,802 of 7,677,245 parameters
(≈ 2.15%)** — and that fraction shrinks further on larger models, where $d_\text{in} d_\text{out}$
grows quadratically but $r\,(d_\text{in}+d_\text{out})$ only linearly.

> **Reference.** Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L.,
> & Chen, W. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.* arXiv:2106.09685
> (ICLR 2022). <https://arxiv.org/abs/2106.09685>

### Why this matters
- Memory: frozen weights need no gradient storage.
- Speed: fewer parameters to update each step.
- Portability: you can share a tiny adapter file (~100 KB) instead of a full
  model checkpoint, and swap adapters at inference time without reloading $W$.
- Quality: often matches full fine-tuning at the same data size.

Runs in a few minutes on CPU with ESM-2 8M. Install the adapter library first:

```bash
pip install peft
```

> **Run order matters.** The cells below build on each other. Run them **top to bottom** (Jupyter: *Run → Run All Cells*; VS Code: *Run All*). If you hit `NameError: name 'torch' is not defined` (or similar), you skipped the **Setup** cell — run it first.

## Setup — imports & configuration

**Run this cell first.** It imports every library and defines the module-level constants the rest of the notebook relies on.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, f1_score
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
DATASET_NAME = "proteinea/solubility"
NUM_LABELS = 2
N_TRAIN = 400
N_TEST = 150
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["query", "value"]

In [ ]:
# MLflow tracking -> shared local SQLite store.
# (repo is pip-installed via `pip install -e .`, so this import needs no sys.path shim)
import mlflow_utils as mu
mlflow = mu.setup_mlflow()

### `ProteinDataset` (class)

Tokenize protein sequences and cache tensors in memory.

In [3]:
class ProteinDataset(Dataset):

    def __init__(self, sequences, labels, tokenizer):
        self.encodings = tokenizer(
            list(sequences),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }

### `train_epoch` (function)

One pass over the training set; returns mean loss.

In [4]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

### `evaluate` (function)

Returns (accuracy, f1) over a DataLoader.

In [5]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch["labels"].cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="binary")
    return acc, f1

### `count_parameters` (function)

Return (trainable, total) parameter counts.

In [6]:
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

### `main` (function)

In [7]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 1. Load tokenizer and base classification model.
    print(f"\nLoading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    )

    # 2. Wrap with LoRA adapters.
    # get_peft_model FREEZES all base weights and inserts A/B matrices into
    # every linear layer whose name matches LORA_TARGET_MODULES.
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",            # don't train bias terms (keeps adapter tiny)
    )
    model = get_peft_model(base_model, lora_config)

    # 3. Print trainable-parameter stats — this is the key LoRA insight.
    trainable, total = count_parameters(model)
    pct = 100.0 * trainable / total
    print(f"\nTrainable parameters: {trainable:,} / {total:,}  ({pct:.2f}% of total)")
    print("  (Full fine-tuning — Lesson 3 — would train all of the above.)")
    model.print_trainable_parameters()   # peft's built-in summary

    model = model.to(device)

    # 4. Load and subset the dataset.
    print(f"\nLoading dataset: {DATASET_NAME}")
    raw = load_dataset(DATASET_NAME)
    raw = raw.rename_columns({"sequences": "sequence", "labels": "label"})
    raw = raw.map(lambda b: {"label": [int(x) for x in b["label"]]}, batched=True)
    raw = raw.shuffle(seed=42)
    train_raw = raw["train"].select(range(N_TRAIN))
    test_raw = raw["test"].select(range(N_TEST))
    print(f"  Train: {len(train_raw)} sequences  |  Test: {len(test_raw)} sequences")

    # 5. Tokenize into PyTorch Datasets.
    print("Tokenizing...")
    train_ds = ProteinDataset(train_raw["sequence"], train_raw["label"], tokenizer)
    test_ds = ProteinDataset(test_raw["sequence"], test_raw["label"], tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    # Only the adapter A/B weights have requires_grad=True, so this optimizer
    # only receives those tensors.
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
    )

    # 6. Training loop — logged as one MLflow run; per-epoch test metrics are
    # streamed with step=epoch so the run shows a learning curve.
    params = {
        "model": MODEL_NAME, "dataset": DATASET_NAME, "method": "LoRA",
        "n_train": N_TRAIN, "n_test": N_TEST, "epochs": EPOCHS,
        "lr": LEARNING_RATE, "batch_size": BATCH_SIZE, "max_len": MAX_LEN,
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "lora_targets": str(LORA_TARGET_MODULES),
        "trainable_params": int(trainable), "total_params": int(total),
        "trainable_pct": round(pct, 4),
    }
    print(f"\nFine-tuning LoRA adapters for {EPOCHS} epoch(s)...")
    with mu.run("plm-solubility", "l7_esm2_8M_lora", params=params,
                tags={"lesson": "plm_l7", "method": "lora"}):
        for epoch in range(1, EPOCHS + 1):
            loss = train_epoch(model, train_loader, optimizer, device)
            acc, f1 = evaluate(model, test_loader, device)
            mlflow.log_metrics({"train_loss": float(loss), "test_acc": float(acc),
                                "test_f1": float(f1)}, step=epoch)
            print(f"  Epoch {epoch}/{EPOCHS}  loss={loss:.4f}  test_acc={acc:.3f}  test_f1={f1:.3f}")

        # 7. Final results.
        acc, f1 = evaluate(model, test_loader, device)
        mlflow.log_metrics({"test_acc": float(acc), "test_f1": float(f1)})
    print(f"\nFinal test results:")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  F1       : {f1:.3f}")
    print(f"  Params trained: {trainable:,} ({pct:.2f}% of {total:,} total)")
    print(
        "  Compare: Lesson 3 (full fine-tune) trains all params for similar accuracy."
    )

    print(
        """
Things to experiment with:
- Vary rank: LORA_R = 2, 4, 16, 32 — lower rank = fewer params but less capacity.
- Increase lora_alpha relative to r (e.g. alpha=32, r=8) for a stronger scaling.
- Add more target modules: LORA_TARGET_MODULES = ["query", "key", "value", "dense"]
- Use a larger model: MODEL_NAME = "facebook/esm2_t12_35M_UR50D" (watch RAM usage).
- Merge adapters into the base weights for zero-overhead inference:
      merged = model.merge_and_unload()
- Try IA3 (even fewer params than LoRA): peft.IA3Config instead of LoraConfig.
"""
    )

## Run the lesson

Execute everything above, then run `main()`.

In [8]:
main()

Using device: cuda

Loading model: facebook/esm2_t6_8M_UR50D


[transformers] EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Trainable parameters: 164,802 / 7,677,245  (2.15% of total)
  (Full fine-tuning — Lesson 3 — would train all of the above.)
trainable params: 164,802 || all params: 7,677,245 || trainable%: 2.1466

Loading dataset: proteinea/solubility


  Train: 400 sequences  |  Test: 150 sequences
Tokenizing...



Fine-tuning LoRA adapters for 3 epoch(s)...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


  Epoch 1/3  loss=0.6786  test_acc=0.580  test_f1=0.376


  Epoch 2/3  loss=0.6586  test_acc=0.593  test_f1=0.396


  Epoch 3/3  loss=0.6012  test_acc=0.573  test_f1=0.360

Final test results:
  Accuracy : 0.573
  F1       : 0.360
  Params trained: 164,802 (2.15% of 7,677,245 total)
  Compare: Lesson 3 (full fine-tune) trains all params for similar accuracy.

Things to experiment with:
- Vary rank: LORA_R = 2, 4, 16, 32 — lower rank = fewer params but less capacity.
- Increase lora_alpha relative to r (e.g. alpha=32, r=8) for a stronger scaling.
- Add more target modules: LORA_TARGET_MODULES = ["query", "key", "value", "dense"]
- Use a larger model: MODEL_NAME = "facebook/esm2_t12_35M_UR50D" (watch RAM usage).
- Merge adapters into the base weights for zero-overhead inference:
      merged = model.merge_and_unload()
- Try IA3 (even fewer params than LoRA): peft.IA3Config instead of LoraConfig.



## Run comparison — this lesson's MLflow history

How this run compares to every prior run of the same lesson (bigger model / more epochs accumulate here as you re-run). Generated from the shared local MLflow store.

In [ ]:
# --- Run comparison -----------------------------------------------------------
# Every run of this lesson logs to the shared MLflow store. This chart shows
# how the latest run compares to all prior ones (e.g. more epochs / a bigger
# model). It accumulates as you re-run; a no-op on a fresh checkout.
import mlflow_utils as mu
mu.plot_run_comparison('plm-solubility')
